# Clusterização: otimizando o resultado dos agrupamentos

Nesta etapa, vamos melhorar o resultado do KMeans escalando os dados entre 0 e 1 com `MinMaxScaler` e reavaliando a quantidade de clusters mais adequada.

## 1. Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

import joblib

## 2. Carregamento dos dados

In [ ]:
url = 'https://raw.githubusercontent.com/alura-cursos/Clusterizacao-dados-sem-rotulo/main/Dados/dados_mkt.csv'

df = pd.read_csv(url)
df.head()

## 3. Análise inicial

In [ ]:
df.info()

df['sexo'].unique()

## 4. Aplicando One-Hot Encoding

In [ ]:
encoder = OneHotEncoder(categories=[['F', 'M', 'NE']], sparse_output=False)

encoded_sexo = encoder.fit_transform(df[['sexo']])
encoded_df = pd.DataFrame(
    encoded_sexo,
    columns=encoder.get_feature_names_out(['sexo'])
)

dados = pd.concat([df, encoded_df], axis=1).drop('sexo', axis=1)

dados.head()

## 5. Salvando o encoder

In [ ]:
joblib.dump(encoder, 'encoder.pkl')

## 6. Estatísticas dos dados antes do escalonamento

In [ ]:
dados.describe()

## 7. Escalando os dados com MinMaxScaler

In [ ]:
scaler = MinMaxScaler()

dados_escalados = scaler.fit_transform(dados)

dados_escalados

## 8. Convertendo os dados escalados para DataFrame

In [ ]:
dados_escalados = pd.DataFrame(
    dados_escalados,
    columns=dados.columns
)

dados_escalados.describe()

## 9. Salvando o scaler treinado

In [ ]:
joblib.dump(scaler, 'scaler.pkl')

## 10. Função de avaliação para diferentes valores de k

In [ ]:
def avaliacao(dados):
    inercia = []
    silhueta = []

    for k in range(2, 21):
        kmeans = KMeans(n_clusters=k, random_state=45, n_init='auto')
        kmeans.fit(dados)

        inercia.append(kmeans.inertia_)
        silhueta.append(
            f'k={k} - {silhouette_score(dados, kmeans.predict(dados))}'
        )

    return silhueta, inercia

## 11. Avaliando os dados escalados

In [ ]:
silhueta, inercia = avaliacao(dados_escalados)

silhueta

## 12. Função para gráfico de silhueta

In [ ]:
def graf_silhueta(n_clusters, dados):
    kmeans = KMeans(n_clusters=n_clusters, random_state=45, n_init='auto')
    cluster_previsoes = kmeans.fit_predict(dados)

    silhueta_media = silhouette_score(dados, cluster_previsoes)
    print(f'Valor médio para {n_clusters} clusters: {silhueta_media:.3f}')

    silhueta_amostra = silhouette_samples(dados, cluster_previsoes)

    fig, ax1 = plt.subplots(1, 1)
    fig.set_size_inches(9, 7)
    ax1.set_xlim([-0.1, 1])
    ax1.set_ylim([0, len(dados) + (n_clusters + 1) * 10])

    y_lower = 10

    for i in range(n_clusters):
        ith_cluster_silhueta_amostra = silhueta_amostra[cluster_previsoes == i]
        ith_cluster_silhueta_amostra.sort()

        tamanho_cluster_i = ith_cluster_silhueta_amostra.shape[0]
        y_upper = y_lower + tamanho_cluster_i

        cor = cm.nipy_spectral(float(i) / n_clusters)

        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhueta_amostra,
            facecolor=cor,
            edgecolor=cor,
            alpha=0.7
        )

        ax1.text(-0.05, y_lower + 0.5 * tamanho_cluster_i, str(i))
        y_lower = y_upper + 10

    ax1.axvline(x=silhueta_media, color='red', linestyle='--')
    ax1.set_title(f'Gráfico da Silhueta para {n_clusters} clusters')
    ax1.set_xlabel('Valores do coeficiente de silhueta')
    ax1.set_ylabel('Rótulo do cluster')

    ax1.set_yticks([])
    ax1.set_xticks([i / 10.0 for i in range(-1, 11)])

    plt.show()

## 13. Gerando o gráfico de silhueta para 3 clusters

In [ ]:
graf_silhueta(3, dados_escalados)

## 14. Função para o método do cotovelo

In [ ]:
def plot_cotovelo(inercia):
    plt.figure(figsize=(8, 4))
    plt.plot(range(2, 21), inercia, 'bo-')
    plt.xlabel('Número de clusters')
    plt.ylabel('Inércia')
    plt.title('Método do Cotovelo para Determinação de k')
    plt.show()

## 15. Gerando o gráfico do método do cotovelo

In [ ]:
plot_cotovelo(inercia)

## 16. Treinando o modelo final com 3 clusters

In [ ]:
modelo_kmeans = KMeans(n_clusters=3, random_state=45, n_init='auto')

modelo_kmeans.fit(dados_escalados)

## 17. Salvando o modelo final

In [ ]:
joblib.dump(modelo_kmeans, 'kmeans.pkl')

## 18. Gerando a base final com os clusters

In [ ]:
df_resultado = df.copy()
df_resultado['cluster'] = modelo_kmeans.labels_

df_resultado.to_csv('dados_clusterizados_escalados.csv', index=False)

df_resultado.head()

## 19. Quantidade de registros por cluster

In [ ]:
df_resultado['cluster'].value_counts()

## Conclusão

Nesta atividade, os dados foram escalados com `MinMaxScaler` para reduzir o impacto de diferenças de escala entre as variáveis. Após o escalonamento, o modelo KMeans foi reavaliado utilizando inércia, Silhouette Score, gráfico de silhueta e método do cotovelo.

Com base nessa análise, foi escolhido `k = 3` como melhor configuração para o agrupamento. O modelo final foi treinado com os dados escalados e salvo com `joblib`, junto com o scaler utilizado no pré-processamento.